# 🧪 Kodomo Exercises: Protein Analysis
# Упражнения Кодомо: Анализ белков

---

Based on MSU Kodomo Bioinformatics Program (pr11)

## 🎯 Learning Objectives

- Calculate amino acid composition
- Decode protein sequences
- Analyze protein properties
- Work with codon tables

---

## Amino Acid Reference

```
┌─────────────────────────────────────────────────────────────────┐
│                  20 STANDARD AMINO ACIDS                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  NONPOLAR (Hydrophobic):                                        │
│  ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐        │
│  │  G  │  A  │  V  │  L  │  I  │  M  │  F  │  W  │  P  │        │
│  │ Gly │ Ala │ Val │ Leu │ Ile │ Met │ Phe │ Trp │ Pro │        │
│  └─────┴─────┴─────┴─────┴─────┴─────┴─────┴─────┴─────┘        │
│                                                                 │
│  POLAR (Uncharged):                                             │
│  ┌─────┬─────┬─────┬─────┬─────┬─────┐                          │
│  │  S  │  T  │  C  │  Y  │  N  │  Q  │                          │
│  │ Ser │ Thr │ Cys │ Tyr │ Asn │ Gln │                          │
│  └─────┴─────┴─────┴─────┴─────┴─────┘                          │
│                                                                 │
│  CHARGED:                                                       │
│  ┌─────┬─────┬─────┬─────┬─────┐                                │
│  │ D-  │ E-  │ K+  │ R+  │ H+  │                                │
│  │ Asp │ Glu │ Lys │ Arg │ His │                                │
│  └─────┴─────┴─────┴─────┴─────┘                                │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Exercise 1: Amino Acid Composition

**Task:** Calculate the frequency of each amino acid in a protein.

Based on `Kravchenko_pr11_aa-composition.py`

In [ ]:
def amino_acid_composition(protein_sequence):
    """
    Calculate amino acid composition of a protein.
    
    Parameters:
        protein_sequence: String of single-letter amino acid codes
        
    Returns:
        Dictionary with AA counts and percentages
    """
    # Standard amino acids
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    
    # Clean sequence
    seq = protein_sequence.upper()
    seq_length = len(seq)
    
    composition = {}
    for aa in amino_acids:
        count = seq.count(aa)
        percentage = (count / seq_length * 100) if seq_length > 0 else 0
        composition[aa] = {'count': count, 'percentage': percentage}
    
    return composition

# Example: Human insulin B chain
insulin_b = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"

print(f"Protein: Human Insulin B chain")
print(f"Sequence: {insulin_b}")
print(f"Length: {len(insulin_b)} residues")
print("\nAmino Acid Composition:")
print("-" * 40)

composition = amino_acid_composition(insulin_b)
for aa, data in sorted(composition.items()):
    if data['count'] > 0:
        print(f"  {aa}: {data['count']:3d} ({data['percentage']:5.1f}%)")

---

## Exercise 2: Protein Properties

**Task:** Calculate molecular weight, charge, and hydrophobicity.

```
┌─────────────────────────────────────────────────────────────────┐
│                AMINO ACID PROPERTIES                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Molecular Weight (Da):                                         │
│  A:89   C:121  D:133  E:147  F:165  G:75   H:155  I:131         │
│  K:146  L:131  M:149  N:132  P:115  Q:146  R:174  S:105         │
│  T:119  V:117  W:204  Y:181                                     │
│                                                                 │
│  Average protein: ~110 Da per amino acid                        │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Amino acid molecular weights (in Daltons)
AA_WEIGHTS = {
    'A': 89.1,  'C': 121.2, 'D': 133.1, 'E': 147.1, 'F': 165.2,
    'G': 75.1,  'H': 155.2, 'I': 131.2, 'K': 146.2, 'L': 131.2,
    'M': 149.2, 'N': 132.1, 'P': 115.1, 'Q': 146.2, 'R': 174.2,
    'S': 105.1, 'T': 119.1, 'V': 117.1, 'W': 204.2, 'Y': 181.2
}

# Hydrophobicity scale (Kyte-Doolittle)
HYDROPHOBICITY = {
    'A': 1.8,  'C': 2.5,  'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5,  'K': -3.9, 'L': 3.8,
    'M': 1.9,  'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2,  'W': -0.9, 'Y': -1.3
}

def calculate_molecular_weight(sequence):
    """
    Calculate protein molecular weight.
    
    Note: Subtract 18 Da for each peptide bond (water loss)
    """
    seq = sequence.upper()
    weight = sum(AA_WEIGHTS.get(aa, 0) for aa in seq)
    # Subtract water for peptide bonds
    weight -= 18.0 * (len(seq) - 1)
    return weight

def calculate_charge(sequence, ph=7.0):
    """
    Estimate protein charge at given pH.
    
    Simplified calculation using counts of charged residues.
    """
    seq = sequence.upper()
    
    # Positive charges: K, R, H (His partially charged at pH 7)
    positive = seq.count('K') + seq.count('R') + 0.1 * seq.count('H')
    # Add N-terminus
    positive += 1
    
    # Negative charges: D, E
    negative = seq.count('D') + seq.count('E')
    # Add C-terminus
    negative += 1
    
    return positive - negative

def calculate_gravy(sequence):
    """
    Calculate GRAVY (Grand Average of Hydropathy).
    
    Positive = hydrophobic, Negative = hydrophilic
    """
    seq = sequence.upper()
    if len(seq) == 0:
        return 0
    total = sum(HYDROPHOBICITY.get(aa, 0) for aa in seq)
    return total / len(seq)

# Analyze insulin B chain
print("Protein Analysis: Insulin B chain")
print("=" * 40)
print(f"Sequence: {insulin_b}")
print(f"Length: {len(insulin_b)} residues")
print(f"Molecular Weight: {calculate_molecular_weight(insulin_b):.1f} Da")
print(f"Net Charge (pH 7): {calculate_charge(insulin_b):+.1f}")
print(f"GRAVY Index: {calculate_gravy(insulin_b):.2f}")

---

## Exercise 3: Codon Table & Translation

**Task:** Translate DNA to protein using the genetic code.

Based on `Kravchenko_pr11_decode.py`

In [ ]:
# Standard genetic code
CODON_TABLE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate(dna_sequence, reading_frame=0):
    """
    Translate DNA sequence to protein.
    
    Parameters:
        dna_sequence: DNA string
        reading_frame: 0, 1, or 2
        
    Returns:
        Protein sequence (stops at stop codon)
    """
    seq = dna_sequence.upper()
    protein = []
    
    for i in range(reading_frame, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = CODON_TABLE.get(codon, 'X')
        if aa == '*':  # Stop codon
            break
        protein.append(aa)
    
    return ''.join(protein)

# Example: Translate a gene fragment
dna = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"

print(f"DNA sequence:")
print(f"5'-{dna}-3'")
print()

print("Translation in all 3 reading frames:")
for frame in range(3):
    protein = translate(dna, frame)
    print(f"  Frame +{frame+1}: {protein}")

---

## Exercise 4: Protein Secondary Structure Prediction

**Task:** Predict secondary structure propensity.

Using Chou-Fasman propensity values (simplified).

In [ ]:
# Simplified Chou-Fasman propensities
HELIX_PROPENSITY = {
    'A': 1.42, 'C': 0.70, 'D': 1.01, 'E': 1.51, 'F': 1.13,
    'G': 0.57, 'H': 1.00, 'I': 1.08, 'K': 1.16, 'L': 1.21,
    'M': 1.45, 'N': 0.67, 'P': 0.57, 'Q': 1.11, 'R': 0.98,
    'S': 0.77, 'T': 0.83, 'V': 1.06, 'W': 1.08, 'Y': 0.69
}

SHEET_PROPENSITY = {
    'A': 0.83, 'C': 1.19, 'D': 0.54, 'E': 0.37, 'F': 1.38,
    'G': 0.75, 'H': 0.87, 'I': 1.60, 'K': 0.74, 'L': 1.30,
    'M': 1.05, 'N': 0.89, 'P': 0.55, 'Q': 1.10, 'R': 0.93,
    'S': 0.75, 'T': 1.19, 'V': 1.70, 'W': 1.37, 'Y': 1.47
}

def predict_structure(sequence, window=5):
    """
    Predict secondary structure using sliding window.
    
    Returns:
        String with H (helix), E (sheet), C (coil) predictions
    """
    seq = sequence.upper()
    prediction = []
    
    for i in range(len(seq)):
        # Get window around position
        start = max(0, i - window // 2)
        end = min(len(seq), i + window // 2 + 1)
        window_seq = seq[start:end]
        
        # Calculate average propensities
        helix_avg = sum(HELIX_PROPENSITY.get(aa, 1.0) for aa in window_seq) / len(window_seq)
        sheet_avg = sum(SHEET_PROPENSITY.get(aa, 1.0) for aa in window_seq) / len(window_seq)
        
        # Assign structure
        if helix_avg > 1.0 and helix_avg > sheet_avg:
            prediction.append('H')
        elif sheet_avg > 1.0 and sheet_avg > helix_avg:
            prediction.append('E')
        else:
            prediction.append('C')
    
    return ''.join(prediction)

# Predict structure for insulin B chain
print("Secondary Structure Prediction")
print("=" * 40)
print(f"Sequence:   {insulin_b}")
prediction = predict_structure(insulin_b)
print(f"Prediction: {prediction}")
print()
print("Legend: H=helix, E=sheet, C=coil")

---

## 🏋️ Practice Exercises

### Exercise A: Isoelectric Point Calculator
Estimate the pI of a protein sequence.

### Exercise B: Transmembrane Helix Predictor
Find regions with high hydrophobicity that might span membranes.

### Exercise C: Extinction Coefficient Calculator
Calculate protein extinction coefficient from W, Y, C content.

---

## 📚 Key Takeaways

1. **Amino acid composition** affects protein properties
2. **Molecular weight** is sum of residue weights minus water
3. **GRAVY index** indicates overall hydrophobicity
4. **Chou-Fasman** propensities predict secondary structure